# Notebook 04: Tier-2 End-to-End ResNet-18 & Dynamic Leaf Expansion

This notebook orchestrates the Feature F5 experiments on remote JupyterHub:
1. Dynamic entropy-based leaf sub-clustering ablation on CIFAR-100.
2. Tier-2 End-to-End ResNet-18 vision pipeline across negative sampling strategies.
3. Periodic Ward Confusion Tree refresh during end-to-end training.

In [ ]:
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from src.utils.seed import set_global_seed
from src.trees.semantic_tree import build_semantic_tree
from src.mcts.expansion import maybe_expand_leaf, expand_leaf_by_entropy, compute_prediction_entropy
from src.training.trainer import train_scenario1
from src.training.e2e_trainer import Tier2ResNet18, train_e2e_scenario1, E2EExperimentConfig, run_e2e_experiment

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Dynamic Entropy Leaf Expansion Ablation

Compare standard MCTS-HNM against MCTS-HNM with dynamic entropy sub-clustering.

In [ ]:
n_samples = 100
feat_dim = 32
features = torch.randn(n_samples, feat_dim)
labels = np.array([0] * 20 + [1] * 40 + [2] * 40, dtype=np.int64)
pos_idx = np.arange(0, 20, dtype=np.int64)
neg_idx = np.arange(20, 100, dtype=np.int64)
bin_labels = (labels == 0).astype(np.int64)

class_to_indices = {1: np.arange(20, 60, dtype=np.int64), 2: np.arange(60, 100, dtype=np.int64)}
tree_static = build_semantic_tree(class_to_indices)
tree_dyn = build_semantic_tree(class_to_indices)

print(f"Initial leaves: {tree_dyn.num_leaves}")
res_static = train_scenario1(
    train_features=features,
    train_binary_labels=bin_labels,
    positive_indices=pos_idx,
    negative_indices=neg_idx,
    test_features=features,
    test_binary_labels=bin_labels,
    strategy="mcts_enhanced",
    tree=tree_static,
    num_steps=30,
    eval_every=10,
    enable_leaf_expansion=False,
)

res_dyn = train_scenario1(
    train_features=features,
    train_binary_labels=bin_labels,
    positive_indices=pos_idx,
    negative_indices=neg_idx,
    test_features=features,
    test_binary_labels=bin_labels,
    strategy="mcts_enhanced",
    tree=tree_dyn,
    num_steps=30,
    eval_every=10,
    enable_leaf_expansion=True,
    split_threshold=5,
    k_sub=2,
)
print(f"Static final F1: {res_static['final_metrics']['f1']:.4f}")
print(f"Dynamic final F1: {res_dyn['final_metrics']['f1']:.4f}, final leaves: {res_dyn['final_num_leaves']}")

## 2. Tier-2 End-to-End ResNet-18 Benchmark

Train ResNet-18 on raw images and evaluate end-to-end performance.

In [ ]:
images = torch.randn(60, 3, 32, 32)
img_labels = np.array([0] * 20 + [1] * 20 + [2] * 20, dtype=np.int64)
img_pos_idx = np.arange(0, 20, dtype=np.int64)
img_neg_idx = np.arange(20, 60, dtype=np.int64)
img_bin_labels = (img_labels == 0).astype(np.int64)

def make_tree():
    return build_semantic_tree({1: np.arange(20, 40, dtype=np.int64), 2: np.arange(40, 60, dtype=np.int64)})

cfg = E2EExperimentConfig(
    scenario=1,
    strategies=("uniform", "mcts_enhanced"),
    seeds=(42,),
    num_steps=10,
    eval_every=5,
    pos_batch_size=2,
    neg_batch_size=4,
    width_multiplier=0.25,
    output_dir="results",
)

summary = run_e2e_experiment(
    config=cfg,
    train_images=images,
    train_labels=img_bin_labels,
    test_images=images,
    test_labels=img_bin_labels,
    positive_indices=img_pos_idx,
    negative_indices=img_neg_idx,
    tree_factory=make_tree,
    device=device,
)
print("Tier-2 E2E Summary:")
for s, data in summary["strategies"].items():
    print(f"  {s}: mean F1 = {data['mean']:.4f}")